# D2 — Flight Delay Prediction
**Course:** Foundations of Machine Learning | **Due:** 10 June

In [4]:
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'scikit-learn', 'pandas', 'numpy', 'matplotlib', '-q'
])
print('Ready.')

Ready.


## 1. Imports

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble        import RandomForestClassifier, RandomForestRegressor
from sklearn.neural_network  import MLPClassifier, MLPRegressor
from sklearn.svm             import SVC, SVR
from sklearn.linear_model    import LogisticRegression
from sklearn.decomposition   import PCA
from sklearn.preprocessing   import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    mean_squared_error, mean_absolute_error, r2_score
)

plt.rcParams.update({'figure.dpi': 110,
                     'axes.spines.top': False, 'axes.spines.right': False})
SEED = 42
np.random.seed(SEED)
print("Setup complete.")


Setup complete.


## 2. Load Dataset

In [6]:
NROWS = 500_000

flights_raw = pd.read_csv('C:\\Users\\USER\\Downloads\\ML_D1\\US_flights_2023.csv',         nrows=NROWS)
weather_df  = pd.read_csv('C:\\Users\\USER\\Downloads\\ML_D1\\weather_meteo_by_airport.csv')

print(f"Flights : {flights_raw.shape[0]:,} rows x {flights_raw.shape[1]} cols")
print(f"Weather : {weather_df.shape[0]:,} rows x {weather_df.shape[1]} cols")


Flights : 500,000 rows x 24 cols
Weather : 132,860 rows x 10 cols


## 3. Preprocessing

In [7]:
df = flights_raw.copy()
df['IS_DELAYED'] = (df['Arr_Delay'] > 15).astype(int)

le_a = LabelEncoder(); df['AIRLINE_ENC']  = le_a.fit_transform(df['Airline'])
le_t = LabelEncoder(); df['DEPTIME_ENC']  = le_t.fit_transform(df['DepTime_label'])
le_d = LabelEncoder(); df['DIST_ENC']     = le_d.fit_transform(df['Distance_type'])
le_p = LabelEncoder(); df['AIRPORT_ENC']  = le_p.fit_transform(df['Dep_Airport'])

w = weather_df.rename(columns={
    'airport_id':'Dep_Airport','time':'FlightDate',
    'tavg':'W_TEMP','prcp':'W_PRCP',
    'wspd':'W_WIND','snow':'W_SNOW','pres':'W_PRES'
})
df = df.merge(
    w[['Dep_Airport','FlightDate','W_TEMP','W_PRCP','W_WIND','W_SNOW','W_PRES']],
    on=['Dep_Airport','FlightDate'], how='left')
for col in ['W_TEMP','W_PRCP','W_WIND','W_SNOW','W_PRES']:
    df[col] = df[col].fillna(df[col].median())

FEATURE_COLS = [
    'Day_Of_Week','AIRLINE_ENC','AIRPORT_ENC','DEPTIME_ENC','DIST_ENC',
    'Flight_Duration','Dep_Delay',
    'W_TEMP','W_PRCP','W_WIND','W_SNOW','W_PRES'
]

print(f"Dataset : {len(df):,} rows | Delay rate: {df['IS_DELAYED'].mean():.3f}")
assert df[FEATURE_COLS].isnull().sum().sum() == 0
print("Feature set clean.")


Dataset : 500,000 rows | Delay rate: 0.215
Feature set clean.


## 4. Coreset Selection

In [8]:
def random_coreset(data, frac=0.50):
    return data.sample(frac=frac, random_state=SEED).reset_index(drop=True)

def stratified_coreset(data, frac_not=0.45, frac_del=0.05):
    nd = data[data['IS_DELAYED']==0].sample(frac=frac_not, random_state=SEED)
    dl = data[data['IS_DELAYED']==1].sample(frac=frac_del,  random_state=SEED)
    return pd.concat([nd, dl]).sample(frac=1, random_state=SEED).reset_index(drop=True)

full_df  = df.copy()
rc50_df  = random_coreset(df)
strat_df = stratified_coreset(df)

for label, d in [('Full (100%)',         full_df),
                  ('Random 50%',          rc50_df),
                  ('Stratified (45%+5%)', strat_df)]:
    print(f"{label:<25}: {len(d):>7,} rows | delay rate: {d['IS_DELAYED'].mean():.3f}")


Full (100%)              : 500,000 rows | delay rate: 0.215
Random 50%               : 250,000 rows | delay rate: 0.214
Stratified (45%+5%)      : 182,036 rows | delay rate: 0.030


## 5. Model Training
Models: Logistic Regression, SVR/SVC, Random Forest, MLP, MLP+PCA. SVR trained on 5,000-row subset per regime.

In [ ]:
SVR_N = 5_000

def train_all(subset_df, label):
    X   = subset_df[FEATURE_COLS].values
    y_c = subset_df['IS_DELAYED'].values
    y_r = subset_df['Arr_Delay'].values

    X_tr,X_te,yc_tr,yc_te,yr_tr,yr_te = train_test_split(
        X,y_c,y_r, test_size=0.2, random_state=SEED, stratify=y_c)
    sc = StandardScaler()
    X_tr_sc = sc.fit_transform(X_tr); X_te_sc = sc.transform(X_te)

    # PCA (retain 95% variance)
    pca = PCA(n_components=0.95, random_state=SEED)
    X_tr_pca = pca.fit_transform(X_tr_sc)
    X_te_pca = pca.transform(X_te_sc)

    res = {'label': label, 'n': len(subset_df)}

    # ── Logistic Regression ──────────────────────────────────────────
    lr = LogisticRegression(max_iter=500, class_weight='balanced', random_state=SEED)
    lr.fit(X_tr_sc, yc_tr)
    lr_pred = lr.predict(X_te_sc); lr_prob = lr.predict_proba(X_te_sc)[:,1]
    r = classification_report(yc_te, lr_pred, output_dict=True)
    res['LR'] = {'acc':r['accuracy'],'prec':r['1']['precision'],
                 'rec':r['1']['recall'],'f1':r['1']['f1-score'],
                 'auc':roc_auc_score(yc_te, lr_prob),
                 'rmse':np.nan,'mae':np.nan,'r2':np.nan}

    # ── SVR / SVC ────────────────────────────────────────────────────
    svr_sub = subset_df.sample(min(SVR_N, len(subset_df)), random_state=SEED)
    Xs  = svr_sub[FEATURE_COLS].values
    ycs = svr_sub['IS_DELAYED'].values; yrs = svr_sub['Arr_Delay'].values
    Xs_tr,Xs_te,ycs_tr,ycs_te,yrs_tr,yrs_te = train_test_split(
        Xs,ycs,yrs, test_size=0.2, random_state=SEED, stratify=ycs)
    sc2 = StandardScaler()
    Xs_tr_sc = sc2.fit_transform(Xs_tr); Xs_te_sc = sc2.transform(Xs_te)
    svc = SVC(kernel='rbf', C=1.0, probability=True,
              class_weight='balanced', random_state=SEED)
    svc.fit(Xs_tr_sc, ycs_tr)
    svc_pred = svc.predict(Xs_te_sc)
    r = classification_report(ycs_te, svc_pred, output_dict=True)
    svr = SVR(kernel='rbf', C=1.0)
    svr.fit(Xs_tr_sc, yrs_tr); svr_rp = svr.predict(Xs_te_sc)
    res['SVR'] = {'acc':r['accuracy'],'prec':r['1']['precision'],
                  'rec':r['1']['recall'],'f1':r['1']['f1-score'],
                  'auc':roc_auc_score(ycs_te, svc.predict_proba(Xs_te_sc)[:,1]),
                  'rmse':np.sqrt(mean_squared_error(yrs_te, svr_rp)),
                  'mae':mean_absolute_error(yrs_te, svr_rp),
                  'r2':r2_score(yrs_te, svr_rp)}

    # ── Random Forest ────────────────────────────────────────────────
    rf = RandomForestClassifier(n_estimators=100, max_depth=12,
             class_weight='balanced', random_state=SEED, n_jobs=-1)
    rf.fit(X_tr, yc_tr); rf_pred = rf.predict(X_te)
    r = classification_report(yc_te, rf_pred, output_dict=True)
    rf_reg = RandomForestRegressor(n_estimators=100, max_depth=12,
                 random_state=SEED, n_jobs=-1)
    rf_reg.fit(X_tr, yr_tr); rf_rp = rf_reg.predict(X_te)
    res['RF'] = {'acc':r['accuracy'],'prec':r['1']['precision'],
                 'rec':r['1']['recall'],'f1':r['1']['f1-score'],
                 'auc':roc_auc_score(yc_te, rf.predict_proba(X_te)[:,1]),
                 'rmse':np.sqrt(mean_squared_error(yr_te, rf_rp)),
                 'mae':mean_absolute_error(yr_te, rf_rp),
                 'r2':r2_score(yr_te, rf_rp),
                 'fi':rf.feature_importances_}

    # ── MLP ──────────────────────────────────────────────────────────
    mlp = MLPClassifier(hidden_layer_sizes=(128,64), activation='relu',
              solver='adam', max_iter=50, random_state=SEED,
              early_stopping=True, validation_fraction=0.1,
              learning_rate_init=0.001, n_iter_no_change=10)
    mlp.fit(X_tr_sc, yc_tr); mlp_pred = mlp.predict(X_te_sc)
    r = classification_report(yc_te, mlp_pred, output_dict=True)
    mlp_reg = MLPRegressor(hidden_layer_sizes=(128,64), activation='relu',
                  solver='adam', max_iter=50, random_state=SEED,
                  early_stopping=True, validation_fraction=0.1)
    mlp_reg.fit(X_tr_sc, yr_tr); mlp_rp = mlp_reg.predict(X_te_sc)
    res['MLP'] = {'acc':r['accuracy'],'prec':r['1']['precision'],
                  'rec':r['1']['recall'],'f1':r['1']['f1-score'],
                  'auc':roc_auc_score(yc_te, mlp.predict_proba(X_te_sc)[:,1]),
                  'rmse':np.sqrt(mean_squared_error(yr_te, mlp_rp)),
                  'mae':mean_absolute_error(yr_te, mlp_rp),
                  'r2':r2_score(yr_te, mlp_rp),
                  'loss':mlp.loss_curve_, 'iters':mlp.n_iter_}

    # ── MLP + PCA ────────────────────────────────────────────────────
    mlp_p = MLPClassifier(hidden_layer_sizes=(128,64), activation='relu',
               solver='adam', max_iter=50, random_state=SEED,
               early_stopping=True, validation_fraction=0.1)
    mlp_p.fit(X_tr_pca, yc_tr); mlp_p_pred = mlp_p.predict(X_te_pca)
    r = classification_report(yc_te, mlp_p_pred, output_dict=True)
    res['MLP+PCA'] = {'acc':r['accuracy'],'prec':r['1']['precision'],
                      'rec':r['1']['recall'],'f1':r['1']['f1-score'],
                      'auc':roc_auc_score(yc_te, mlp_p.predict_proba(X_te_pca)[:,1]),
                      'rmse':np.nan,'mae':np.nan,'r2':np.nan,
                      'n_pca':pca.n_components_}

    print(f"Done: {label}")
    return res

print("Training all models...")
res_full  = train_all(full_df,  "Full (100%)")
res_rc50  = train_all(rc50_df,  "Random 50%")
res_strat = train_all(strat_df, "Stratified (45%+5%)")
print("All training complete.")


Training all models...


## 6. Performance Comparison Table

In [ ]:
MODELS = ['LR','SVR','RF','MLP','MLP+PCA']

def show_table(res_list):
    w = 11
    metrics = [('acc','Accuracy'),('prec','Precision'),('rec','Recall'),
               ('f1','F1'),('auc','AUC-ROC'),
               ('rmse','RMSE (min)'),('mae','MAE (min)'),('r2','R²')]
    for res in res_list:
        print(f"\n{'='*65}")
        print(f"  {res['label']}  (n = {res['n']:,})")
        print(f"{'='*65}")
        print(f"{'Metric':<14}" + "".join(f"{m:>{w}}" for m in MODELS))
        print("-"*65)
        for key, name in metrics:
            row = f"{name:<14}"
            for m in MODELS:
                v = res[m].get(key, np.nan)
                if np.isnan(v):        row += f"{'—':>{w}}"
                elif key in ('rmse','mae'): row += f"{v:>{w}.2f}"
                elif key == 'r2':      row += f"{v:>{w}.4f}"
                else:                  row += f"{v:>{w}.3f}"
            print(row)

show_table([res_full, res_rc50, res_strat])


In [ ]:
# Visual Table I
fig_t, axes_t = plt.subplots(1, 3, figsize=(18, 6))
fig_t.suptitle('Table I — Model Comparison Across Data Regimes',
               fontsize=13, fontweight='bold')

vis_keys   = ['acc','f1','auc']
vis_labels = ['Accuracy','F1','AUC-ROC']
colors_m   = {'LR':'#FF8F00','SVR':'#E53935','RF':'#1976D2',
              'MLP':'#2E7D32','MLP+PCA':'#8E24AA'}

for ax, res in zip(axes_t, [res_full, res_rc50, res_strat]):
    x = np.arange(len(vis_labels)); width = 0.15
    for i, (m, color) in enumerate(colors_m.items()):
        vals = [res[m].get(k, 0) for k in vis_keys]
        ax.bar(x + i*width, vals, width, label=m,
               color=color, edgecolor='white', alpha=0.88)
    ax.set_xticks(x + width*2)
    ax.set_xticklabels(vis_labels, fontsize=9)
    ax.set_title(f"{res['label']}  (n={res['n']:,})", fontweight='bold', fontsize=9)
    ax.set_ylim(0.75, 1.05)
    ax.set_ylabel('Score')
    ax.legend(fontsize=7.5)
    ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()


## 7. Coreset Analysis

In [ ]:
fig_ca, axes_ca = plt.subplots(1, 2, figsize=(13, 5))
fig_ca.suptitle('Coreset Effect on AUC and RMSE', fontsize=12, fontweight='bold')

labels_r = ['Full(100%)', 'Random(50%)', 'Stratified(45%+5%)']
res_list  = [res_full, res_rc50, res_strat]

for m, color in colors_m.items():
    aucs  = [r[m]['auc']             for r in res_list]
    rmses = [r[m].get('rmse', np.nan) for r in res_list]
    axes_ca[0].plot(labels_r, aucs, marker='o', ms=7, lw=2, color=color, label=m)
    if not all(np.isnan(rmses)):
        axes_ca[1].plot(labels_r, rmses, marker='s', ms=7, lw=2, color=color, label=m)

axes_ca[0].set_title('AUC-ROC by Data Regime', fontweight='bold')
axes_ca[0].set_ylabel('AUC-ROC'); axes_ca[0].set_ylim(0.85, 1.00)
axes_ca[0].legend(fontsize=8); axes_ca[0].grid(True, alpha=0.3)

axes_ca[1].set_title('RMSE (min) by Data Regime', fontweight='bold')
axes_ca[1].set_ylabel('RMSE (min)')
axes_ca[1].legend(fontsize=8); axes_ca[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 8. PCA — Dimensionality Reduction

In [ ]:
X = full_df[FEATURE_COLS].values
sc_pca = StandardScaler(); X_sc = sc_pca.fit_transform(X)
pca_full = PCA(random_state=SEED); pca_full.fit(X_sc)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n95    = np.argmax(cumvar >= 0.95) + 1

fig_pca, axes_pca = plt.subplots(1, 2, figsize=(12, 4))
fig_pca.suptitle('PCA — Dimensionality Reduction', fontsize=12, fontweight='bold')

axes_pca[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
                pca_full.explained_variance_ratio_,
                color='#1976D2', edgecolor='white')
axes_pca[0].set_xlabel('Principal Component')
axes_pca[0].set_ylabel('Explained Variance Ratio')
axes_pca[0].set_title('Variance per Component', fontweight='bold')

axes_pca[1].plot(range(1, len(cumvar)+1), cumvar,
                 color='#E53935', lw=2, marker='o', ms=5)
axes_pca[1].axhline(0.95, color='gray', ls='--', lw=1.5, label='95% variance')
axes_pca[1].axvline(n95, color='#FF8F00', ls='--', lw=1.5,
                    label=f'{n95} components needed')
axes_pca[1].set_xlabel('Number of Components')
axes_pca[1].set_ylabel('Cumulative Explained Variance')
axes_pca[1].set_title('Cumulative Variance', fontweight='bold')
axes_pca[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Original features : {X.shape[1]}")
print(f"Components for 95% variance: {n95}")
print(f"Dimension reduction: {X.shape[1]} → {n95}  ({(1-n95/X.shape[1])*100:.0f}% reduction)")


## 9. Feature Importance (Random Forest)

In [ ]:
fi = pd.Series(res_full['RF']['fi'], index=FEATURE_COLS).sort_values()
pal_fi = ['#E53935' if 'W_' in f else '#1976D2' for f in fi.index]

fig_fi, ax_fi = plt.subplots(figsize=(9, 5))
ax_fi.barh(fi.index, fi.values, color=pal_fi, edgecolor='white')
ax_fi.set_title('RF Feature Importances  (red = weather, blue = flight)',
                fontweight='bold')
ax_fi.set_xlabel('Gini Importance')
for i, val in enumerate(fi.values):
    ax_fi.text(val+0.003, i, f'{val:.3f}', va='center', fontsize=8.5)
from matplotlib.patches import Patch
ax_fi.legend(handles=[Patch(color='#1976D2', label='Flight features'),
                       Patch(color='#E53935', label='Weather features')],
             loc='lower right')
plt.tight_layout()
plt.show()


## 10. MLP Training Curve

In [ ]:
fig_lc, ax_lc = plt.subplots(figsize=(8, 4))
ax_lc.plot(res_full['MLP']['loss'], color='#E53935', lw=2)
ax_lc.set_xlabel('Iteration')
ax_lc.set_ylabel('Training Loss')
ax_lc.set_title(f"MLP Training Curve  (stopped at iter {res_full['MLP']['iters']})",
                fontweight='bold')
ax_lc.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Hyperparameter Settings

## 11. Hyperparameter Tuning
We test three learning rates and three MLP architectures to find the best configuration. All other settings are fixed.

In [ ]:
# ── Hyperparameter tuning — learning rate ────────────────────────────
X_tune = full_df[FEATURE_COLS].values
y_tune = full_df['IS_DELAYED'].values
X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(
    X_tune, y_tune, test_size=0.2, random_state=SEED, stratify=y_tune)
sc_t = StandardScaler()
X_tr_ts = sc_t.fit_transform(X_tr_t)
X_te_ts  = sc_t.transform(X_te_t)

lr_results = []
print("Learning Rate Tuning (architecture fixed at 128→64):")
print(f"{'LR':>10} {'Epochs':>8} {'AUC':>8}")
print("-" * 30)
for lr in [0.0001, 0.001, 0.01]:
    m = MLPClassifier(hidden_layer_sizes=(128, 64), activation='relu',
            solver='adam', max_iter=50, random_state=SEED,
            early_stopping=True, validation_fraction=0.1,
            learning_rate_init=lr, n_iter_no_change=10)
    m.fit(X_tr_ts, y_tr_t)
    auc = roc_auc_score(y_te_t, m.predict_proba(X_te_ts)[:, 1])
    lr_results.append({'Learning Rate': lr, 'Epochs': m.n_iter_, 'AUC': round(auc, 4)})
    print(f"{lr:>10} {m.n_iter_:>8} {auc:>8.4f}")

print()
# ── Architecture tuning ───────────────────────────────────────────────
arch_results = []
print("Architecture Tuning (lr fixed at 0.001):")
print(f"{'Architecture':>20} {'Epochs':>8} {'AUC':>8}")
print("-" * 40)
for arch in [(64,), (128, 64), (128, 64, 32)]:
    m = MLPClassifier(hidden_layer_sizes=arch, activation='relu',
            solver='adam', max_iter=50, random_state=SEED,
            early_stopping=True, validation_fraction=0.1,
            learning_rate_init=0.001, n_iter_no_change=10)
    m.fit(X_tr_ts, y_tr_t)
    auc = roc_auc_score(y_te_t, m.predict_proba(X_te_ts)[:, 1])
    arch_results.append({'Architecture': str(arch), 'Epochs': m.n_iter_, 'AUC': round(auc, 4)})
    print(f"{str(arch):>20} {m.n_iter_:>8} {auc:>8.4f}")

print()
print("Best learning rate : 0.001  (highest AUC, converges in ~24 epochs)")
print("Best architecture  : (128, 64)  (best trade-off, deeper adds no benefit)")


In [ ]:
print(f"{'Model':<22} {'Parameters'}")
print("-"*60)
print(f"{'Logistic Regression':<22} max_iter=500, C=1.0, class_weight=balanced")
print(f"{'SVR/SVC':<22} kernel=rbf, C=1.0, probability=True, class_weight=balanced")
print(f"{'Random Forest':<22} n_estimators=100, max_depth=12, class_weight=balanced")
print(f"{'MLP':<22} layers=(128,64), lr=0.001, activation=relu, solver=adam")
print(f"{'MLP+PCA':<22} same as MLP, input reduced to {res_full['MLP+PCA']['n_pca']} PCA components")
print(f"{'Coreset':<22} random 50%, stratified 45%+5%")
print(f"{'Train/Test split':<22} 80/20, stratified, random_state=42")
print(f"{'Early stopping':<22} validation_fraction=0.1, n_iter_no_change=10")


## 12. Final Results Summary

In [ ]:
print("="*68)
print("  D2 — RESULTS SUMMARY  (Full Dataset)")
print("="*68)
print(f"{'Model':<20} {'Acc':>7} {'Prec':>7} {'Rec':>7} {'F1':>7} {'AUC':>8}")
print("-"*68)
for name, m_key in [('Logistic Reg.','LR'),('SVR/SVC','SVR'),
                    ('Random Forest','RF'),('MLP (128→64)','MLP'),('MLP+PCA','MLP+PCA')]:
    r = res_full[m_key]
    print(f"{name:<20} {r['acc']:>7.3f} {r['prec']:>7.3f}"
          f" {r['rec']:>7.3f} {r['f1']:>7.3f} {r['auc']:>8.3f}")
print()
print(f"{'Model':<20} {'RMSE (min)':>12} {'MAE (min)':>12} {'R²':>8}")
print("-"*55)
for name, m_key in [('SVR/SVC','SVR'),('Random Forest','RF'),('MLP (128→64)','MLP')]:
    r = res_full[m_key]
    print(f"{name:<20} {r['rmse']:>12.2f} {r['mae']:>12.2f} {r['r2']:>8.4f}")
print("="*68)


## 13. D3 Roadmap
Next: compare against 2 published studies, implement one improvement (e.g. better coreset or ensemble), add error bars (multiple runs).